In [42]:
import requests
import json

def get_runner_ids(eventCode="M2024", place1=48000, place2=48010):
    url = 'https://rmsprodapi.nyrr.org/api/v2/runners/finishers-filter'
    header = {"content-type": "application/json;charset=UTF-8"}
    j = {"eventCode": eventCode, "overallPlaceFrom": place1, "overallPlaceTo": place2,
         "sortColumn": "overallTime", "sortDescending":"false"
}
    response = requests.post(url, headers=header, json=j, allow_redirects=True)
    assert response.status_code == 200
    print(response)
    d = json.loads(response.text)
    print(d["totalItems"], len(d["items"]), d["items"][0]["runnerId"])
    return d

def get_splits(runner_id=42948497):
    url = 'https://rmsprodapi.nyrr.org/api/v2/runners/resultDetails'
    header = {"content-type": "application/json;charset=UTF-8"}
    response = requests.post(url, headers=header, json={"runnerId": str(runner_id)}, allow_redirects=True)
    assert response.status_code == 200, f"Response Code = {response.status_code}"
    d = json.loads(response.text)
    assert d["details"], f"No data specified, d = {d}"
    return d

In [ ]:
get_runner_ids(eventCode="M2023", place1=1, place2=5)

int

In [43]:
get_splits(runner_id=43928778)

AssertionError: No data specified, d = {'details': None, 'success': True, 'message': None}

In [44]:
import time
import pandas as pd
feats = ["runnerId", "bib", "firstName", "lastName", "age", "gender", "city", "countryCode"]

def load_data(last_runner=522, eventCode="M2024"):
    tables = []
    for i in range(1, last_runner, 100):
        place1, place2 = i, i+99
        data = get_runner_ids(eventCode, place1, place2)
        table = pd.DataFrame(data['items'])[feats]
        tables.append(table)
        if (i - 1) % 1000 == 0:
            time.sleep(1)
            print("loaded", i - 1)

    return tables

In [ ]:
tables = load_data(last_runner=47745, eventCode="M2022")
pd.concat(tables)

In [5]:
# pd.concat(tables).to_csv("raw_data/nyc/runnerData22.csv", index=False)

In [137]:
# ids = pd.read_csv("raw_data/nyc/runnerData24.csv")["runnerId"]

In [45]:
data24 = pd.read_csv("raw_data/nyc/runnerData23.csv")
ids = list(data24["runnerId"])


num_runners = 1000
tables2 = []

for j, id in enumerate(ids):
#   try:
    i = j + 1
    splits = get_splits(id)['details']['splitResults']
    dt = {dct["splitCode"]: dct["time"] for dct in splits}
    ser = pd.Series(dt)
    ser["id"] = id
    tables2.append(ser)
    # tables2.append(dt)
    if i % 50 == 0:
        print("loaded", i)
        time.sleep(2)
    if i == num_runners:
        break
#   except Exception as e:
#      print(i, e)
#      continue


loaded 50
loaded 100
loaded 150
loaded 200


AssertionError: Response Code = 429

In [46]:
i

234

In [15]:
s = pd.Series({dct["splitCode"]: dct["time"] for dct in get_splits(id)['details']['splitResults']})
s['id']

5K       0:18:38
10K      0:37:09
15K      0:55:29
20K      1:13:54
HALF     1:17:57
25K      1:32:41
30K      1:51:19
20M      1:59:58
35K      2:11:26
40K      2:32:25
25.2M    2:34:45
26M      2:40:02
MAR      2:41:25
dtype: object

In [5]:
pd.DataFrame(tables2)

,5K,10K,15K,20K,HALF,25K,30K,20M,35K,40K,26M,MAR,id,25.2M
0,0:15:29,0:30:40,0:45:04,0:59:34,1:02:45,1:14:15,1:28:22,1:34:38,1:42:51,1:58:08,2:03:53,2:04:58,42307341,NaN
1,0:15:28,0:30:38,0:45:04,0:59:34,1:02:46,1:14:15,1:28:48,1:35:29,1:44:05,2:00:06,2:05:54,2:06:57,42311423,NaN
2,0:15:31,0:30:40,0:45:06,0:59:39,1:02:52,1:14:37,1:29:07,1:35:42,1:44:24,2:00:20,2:06:08,2:07:11,42307340,2:02:07
3,0:15:29,0:30:38,0:45:05,0:59:36,1:02:47,1:14:34,1:29:40,1:36:37,1:45:40,2:03:23,2:09:11,2:10:21,42317876,2:05:14
4,0:15:31,0:30:37,0:45:29,1:00:33,1:03:56,1:16:28,1:31:48,1:38:43,1:47:42,2:03:35,2:09:21,2:10:25,42304333,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
707,0:20:49,0:41:12,1:01:44,1:22:29,1:27:04,1:43:34,2:04:01,2:13:17,2:25:09,2:47:26,2:55:22,2:56:52,42325948,2:49:54
708,0:20:47,0:41:51,1:02:58,1:24:13,1:28:55,1:46:03,2:07:13,2:16:10,2:28:01,2:48:46,2:55:39,2:56:53,42307211,2:50:57
709,0:19:28,0:39:12,0:59:05,1:19:13,1:23:42,1:40:12,2:00:49,2:10:22,2:22:57,2:46:20,2:55:10,2:56:53,42315503,2:48:55
710,0:20:36,0:41:19,1:02:04,1:22:57,1:27:33,1:44:13,2:04:50,2:14:11,2:26:17,2:47:34,2:55:26,2:56:53,42329729,2:50:00


In [18]:
ids[0], 43091563

(42948497, 43091563)

In [169]:
d1 = data24.iloc[:50, :]
d2 = pd.DataFrame(tables2)
pd.concat([d1, d2], axis=1)

,runnerId,bib,firstName,lastName,age,gender,city,countryCode,5K,10K,...,20K,HALF,25K,30K,20M,35K,40K,25.2M,26M,MAR
0,42948497,7.0,Abdi,Nageeye,35,M,Nijmegen,NLD,0:16:02,0:31:28,...,1:02:21,1:05:35,1:17:48,1:31:58,1:38:17,1:46:20,2:01:10,2:02:52,2:06:36,2:07:39
1,42953047,3.0,Evans,Chebet,35,M,Kapsabet,KEN,0:16:00,0:31:29,...,1:02:20,1:05:34,1:17:48,1:31:57,1:38:17,1:46:20,2:01:10,2:02:52,2:06:39,2:07:45
2,42943555,2.0,Albert,Korir,30,M,Kapkitony,KEN,0:16:00,0:31:28,...,1:02:20,1:05:34,1:17:48,1:31:57,1:38:18,1:46:28,2:01:26,2:03:05,2:06:55,2:08:00
3,42940461,1.0,Tamirat,Tola,33,M,Addis Ababa,ETH,0:16:02,0:31:28,...,1:02:20,1:05:34,1:17:48,1:31:57,1:38:17,1:46:25,2:01:24,2:03:06,2:07:04,2:08:12
4,42934706,6.0,Geoffrey,Kamworor,31,M,Kapchorwa District,KEN,0:16:01,0:31:29,...,1:02:20,1:05:34,1:17:48,1:31:57,1:38:17,1:46:21,2:01:35,2:03:24,2:07:39,2:08:50
5,42950271,10.0,Conner,Mantz,27,M,Provo,USA,0:15:59,0:31:30,...,1:02:24,1:05:35,1:17:48,1:32:14,1:38:53,1:47:24,2:02:38,2:04:17,2:07:58,2:09:00
6,42943637,11.0,Clayton,Young,31,M,Provo,USA,0:16:01,0:31:30,...,1:02:23,1:05:36,1:17:49,1:32:31,1:39:09,1:47:35,2:02:41,2:04:23,2:08:15,2:09:21
7,42964374,9.0,Abel,Kipchumba,30,M,Iten,KEN,0:15:59,0:31:28,...,1:02:19,1:05:35,1:17:49,1:32:15,1:38:56,1:47:34,2:03:28,2:05:21,2:09:33,2:10:39
8,42983034,4.0,Bashir,Abdi,35,M,Ghent,BEL,0:16:03,0:31:31,...,1:02:22,1:05:34,1:17:49,1:32:44,1:39:33,1:48:12,2:03:56,2:05:41,2:09:36,2:10:39
9,42983033,20.0,Cj,Albertson,31,M,Fresno,USA,0:16:02,0:31:30,...,1:02:17,1:05:39,1:17:55,1:32:51,1:39:44,1:48:27,2:04:13,2:05:58,2:09:54,2:10:57


In [167]:
pd.concat([d1, d2], axis=1)

,runnerId,bib,firstName,lastName,age,gender,city,countryCode,5K,10K,...,20K,HALF,25K,30K,20M,35K,40K,25.2M,26M,MAR
0,42948497,7.0,Abdi,Nageeye,35,M,Nijmegen,NLD,0:16:02,0:31:28,...,1:02:21,1:05:35,1:17:48,1:31:58,1:38:17,1:46:20,2:01:10,2:02:52,2:06:36,2:07:39
1,42953047,3.0,Evans,Chebet,35,M,Kapsabet,KEN,0:16:00,0:31:29,...,1:02:20,1:05:34,1:17:48,1:31:57,1:38:17,1:46:20,2:01:10,2:02:52,2:06:39,2:07:45
2,42943555,2.0,Albert,Korir,30,M,Kapkitony,KEN,0:16:00,0:31:28,...,1:02:20,1:05:34,1:17:48,1:31:57,1:38:18,1:46:28,2:01:26,2:03:05,2:06:55,2:08:00
3,42940461,1.0,Tamirat,Tola,33,M,Addis Ababa,ETH,0:16:02,0:31:28,...,1:02:20,1:05:34,1:17:48,1:31:57,1:38:17,1:46:25,2:01:24,2:03:06,2:07:04,2:08:12
4,42934706,6.0,Geoffrey,Kamworor,31,M,Kapchorwa District,KEN,0:16:01,0:31:29,...,1:02:20,1:05:34,1:17:48,1:31:57,1:38:17,1:46:21,2:01:35,2:03:24,2:07:39,2:08:50
5,42950271,10.0,Conner,Mantz,27,M,Provo,USA,0:15:59,0:31:30,...,1:02:24,1:05:35,1:17:48,1:32:14,1:38:53,1:47:24,2:02:38,2:04:17,2:07:58,2:09:00
6,42943637,11.0,Clayton,Young,31,M,Provo,USA,0:16:01,0:31:30,...,1:02:23,1:05:36,1:17:49,1:32:31,1:39:09,1:47:35,2:02:41,2:04:23,2:08:15,2:09:21
7,42964374,9.0,Abel,Kipchumba,30,M,Iten,KEN,0:15:59,0:31:28,...,1:02:19,1:05:35,1:17:49,1:32:15,1:38:56,1:47:34,2:03:28,2:05:21,2:09:33,2:10:39
8,42983034,4.0,Bashir,Abdi,35,M,Ghent,BEL,0:16:03,0:31:31,...,1:02:22,1:05:34,1:17:49,1:32:44,1:39:33,1:48:12,2:03:56,2:05:41,2:09:36,2:10:39
9,42983033,20.0,Cj,Albertson,31,M,Fresno,USA,0:16:02,0:31:30,...,1:02:17,1:05:39,1:17:55,1:32:51,1:39:44,1:48:27,2:04:13,2:05:58,2:09:54,2:10:57


In [82]:
marks = ["5K", "10K", "15K", "20K", "HALF", "25K", "30K", "35K", "40K", "MAR"]

In [10]:
d2 = get_runner_ids(230, 240)
a1 = [c["lastName"] for c in d2["items"]]
a1
# d

<Response [200]>


IndexError: list index out of range

In [11]:
d2

{'totalItems': 0, 'items': []}

In [9]:
test_id =d2['items'][0]["runnerId"]
d3 = get_splits(runner_id = test_id)
d3

IndexError: list index out of range

In [52]:
d3["details"]

{'runnerId': 42939236,
 'bib': '1300',
 'teamName': None,
 'iaaf': 'USA',
 'placeOverall': 230,
 'gunPlaceOverall': 237,
 'netPlaceOverall': 230,
 'timeOverall': '2:34:54',
 'gunTime': '2:35:34',
 'netTime': '2:34:54',
 'scoreByNetTime': True,
 'pace': '05:55',
 'placeGender': 209,
 'placeAgeGroup': 67,
 'ageGroupFromTo': '30-34',
 'timeAgeGrade': '2:34:54',
 'placeAgeGrade': 456,
 'percentAgeGrade': 79.38,
 'placeCountry': 137,
 'speed': 10.2,
 'photoUrl': 'https://www.marathonfoto.com/In?RaceOID=27042024F2&LastName=Harper&BibNumber=1300',
 'basnoPhotoUrl': '',
 'splitResults': [{'splitCode': '5K',
   'splitName': None,
   'time': '0:18:29',
   'pace': '05:57',
   'speed': 10.1,
   'distance': 3.11},
  {'splitCode': '10K',
   'splitName': None,
   'time': '0:36:37',
   'pace': '05:54',
   'speed': 10.2,
   'distance': 6.21},
  {'splitCode': '15K',
   'splitName': None,
   'time': '0:54:51',
   'pace': '05:54',
   'speed': 10.2,
   'distance': 9.32},
  {'splitCode': '20K',
   'splitNam

In [35]:
import json
# url = 'https://rmsprodapi.nyrr.org/api/v2/runners/eventRunner'
url = 'https://rmsprodapi.nyrr.org/api/v2/runners/resultDetails'
j = {"eventCode": "M2024", "bib": "4"}
header = {"content-type": "application/json;charset=UTF-8"}
response = requests.post(url, headers=header, json=j, allow_redirects=True)
print(response)

<Response [200]>


In [23]:
json.loads(response.text)["details"]

{'runnerId': 42948497,
 'bib': '7',
 'teamName': 'NIKE',
 'iaaf': 'NED',
 'placeOverall': 1,
 'gunPlaceOverall': 1,
 'netPlaceOverall': 1,
 'timeOverall': '2:07:39',
 'gunTime': '2:07:39',
 'netTime': '2:07:39',
 'scoreByNetTime': True,
 'pace': '04:53',
 'placeGender': 1,
 'placeAgeGroup': 1,
 'ageGroupFromTo': '35-39',
 'timeAgeGrade': '2:06:57',
 'placeAgeGrade': 1,
 'percentAgeGrade': 96.86,
 'placeCountry': 1,
 'speed': 12.3,
 'photoUrl': 'https://www.marathonfoto.com/In?RaceOID=27042024F2&LastName=Nageeye&BibNumber=7',
 'basnoPhotoUrl': '',
 'splitResults': [{'splitCode': '5K',
   'splitName': None,
   'time': '0:16:02',
   'pace': '05:10',
   'speed': 11.6,
   'distance': 3.11},
  {'splitCode': '10K',
   'splitName': None,
   'time': '0:31:28',
   'pace': '05:04',
   'speed': 11.9,
   'distance': 6.21},
  {'splitCode': '15K',
   'splitName': None,
   'time': '0:46:55',
   'pace': '05:02',
   'speed': 11.9,
   'distance': 9.32},
  {'splitCode': '20K',
   'splitName': None,
   'ti